# Model B, B1 — build fix 1: crossing style fit per attempt (Oct 30, 2026) — EXPLORATORY

A build fix of B1 (definition match), not a new version: `results/model_b_log.md`, entry 1. v1 record: `notebooks/path1/pB1_model.ipynb`.

**Spec (as given):**

B1 fixes (before any R = 200 run):
1. pi: option (a). Refit pi, rho and q_sg on ALL crossing attempts (state C segments, completed or aborted) with a per-player mixture model:
   - each attempt is smooth (stop chance 0) or stop-and-go (q_sg per step);
   - the player's smooth propensity ~ Beta(pi, rho);
   - likelihood of an attempt with n counted steps and k stops: smooth only if k = 0 (likelihood 1); stop-and-go: q_sg^k (1 - q_sg)^(n - k);
   - attempts censored by the round end are included the same way.

   Maximum likelihood per interface, group-bootstrap 95% CIs (1,000, stratified, seed 573). Update pB1_params.csv (keep the old values as pB1_params_v1.csv).
2. old CL rho: a known limitation (14 players, 2 groups); keep the estimate, add a parametric CI, and note it in the notebook and the log.
3. Rerun the B1 recovery once with the new parameters (G6train, R = 2, seed 573): re-estimate pi, rho and q_sg with the NEW per-attempt code. Same pass rule. Report.
4. Log this as a B1 build fix (definition match: fit per attempt), not a new version.

## Choices the spec leaves open

Code: `model_b1.attempt_table`, `fit_attempts`, `boot_attempts`, `simulate_attempts`.

- **Attempt** = the counted moving steps in state C of one outbound segment (B0 labels: outbound, no reversal since leaving the band), whatever the segment's outcome (completed, aborted, censored). n = those steps, k = the stops among them. A segment with no counted C step has no attempt (it gives no information on the style).
- **Players** = (group, seat), all players with >= 1 attempt (no minimum: the mixture weighs each player by its own attempts).
- **Likelihood**, per player: the integral over p ~ Beta(a, b) of the product over its attempts of [p 1{k = 0} + (1 - p) q^k (1 - q)^(n - k)], computed exactly (a polynomial in p; Beta functions).
- **Bootstrap:** files resampled within dataset with `motion.boot_ratios`'s draws (1,000, seed 573); each resample refit from the full-data estimate.
- **Old CL rho, parametric CI:** 1,000 data sets drawn from the fitted mixture (old CL pi, rho, q_sg) with the human players' own attempts (each player's list of n), each refit with the same code (seed 573); the 2.5 / 97.5% points. The estimate is unchanged; the group-bootstrap CI is shown next to it.
- **Recovery:** as in v1 (G6train, R = 2, seed 573, the new pB1_params.csv). pi, rho, q_sg are re-estimated with the per-attempt code, q_E and lam_resume as in v1. PASS = inside the 95% CI (old CL rho: the parametric CI).

In [1]:
import numpy as np
import pandas as pd

from slider import config, design, model_b1, motion, sim, stop_states
from slider import slider_logs as sl

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

runs, table = sl.load_runs("behavior")                  # roles={"train"}: G-train only, no G-val, Ring or CT
assert set(table["role"]) == {"train"}
steps = model_b1.step_state(stop_states.step_table(runs))
att = model_b1.attempt_table(steps)
att.groupby("interface").agg(attempts=("n", "size"), players=("player", lambda p: att.loc[p.index, ["file", "player"]]
                             .drop_duplicates().shape[0]), steps=("n", "sum"), no_stop_share=("k", lambda k: (k == 0).mean()),
                             completed=("outcome", lambda o: (o == "completed").sum()),
                             aborted=("outcome", lambda o: (o == "aborted").sum()),
                             censored=("outcome", lambda o: (o == "censored").sum())).loc[motion.INTERFACES]

,attempts,players,steps,no_stop_share,completed,aborted,censored
interface,,,,,,,
lab_v09,1735,134,21885,0.349856,1267,467,1
old_cl,178,14,9811,0.134831,101,75,2
cc,1974,115,28117,0.369807,1665,299,10


## 1. pi, rho, q_sg per attempt (maximum likelihood; 95% CI: 1,000 stratified group resamples, seed 573)

In [2]:
fits, boots = {}, {}
for name in motion.INTERFACES:
    fits[name], boots[name] = model_b1.boot_attempts(att[att["interface"] == name])
style = pd.DataFrame([{"interface": name, "parameter": par, "estimate": fits[name][i],
                       "ci_low": np.percentile(boots[name][:, i], 2.5), "ci_high": np.percentile(boots[name][:, i], 97.5)}
                      for name in motion.INTERFACES for i, par in enumerate(["pi", "rho", "q_sg"])])
style.round(4)

,interface,parameter,estimate,ci_low,ci_high
0,lab_v09,pi,0.3404,0.3031,0.3787
1,lab_v09,rho,0.1701,0.1069,0.2420
2,lab_v09,q_sg,0.4768,0.4173,0.5311
3,old_cl,pi,0.1618,0.0769,0.3044
4,old_cl,rho,0.1898,0.0000,0.1898
5,old_cl,q_sg,0.1406,0.1207,0.2059
6,cc,pi,0.3202,0.2819,0.3576
7,cc,rho,0.3935,0.3476,0.4420
8,cc,q_sg,0.5014,0.4241,0.5674


## 2. Old CL rho: parametric CI (a known limitation: 14 players, 2 groups)

In [3]:
old = att[att["interface"] == "old_cl"]
players_n = [a["n"].to_numpy() for _, a in old.groupby(["file", "player"])]
rng = np.random.default_rng(config.SEED)
pi0, rho0, q0 = fits["old_cl"]
refit = np.array([model_b1.fit_attempts(model_b1.simulate_attempts(players_n, pi0, rho0, q0, rng)) for _ in range(1000)])
rho_ci = np.percentile(refit[:, 1], [2.5, 97.5])
r = style.set_index(["interface", "parameter"]).loc[("old_cl", "rho")]
print(f"old CL: {len(players_n)} players, {len(old)} attempts, {old['file'].nunique()} groups")
print(f"rho = {rho0:.4f}; group-bootstrap CI [{r['ci_low']:.4f}, {r['ci_high']:.4f}]; "
      f"parametric CI [{rho_ci[0]:.4f}, {rho_ci[1]:.4f}] (1,000 refits, median {np.median(refit[:, 1]):.4f})")

old CL: 14 players, 178 attempts, 2 groups
rho = 0.1898; group-bootstrap CI [0.0000, 0.1898]; parametric CI [0.0000, 1.0000] (1,000 refits, median 0.0000)


## New pB1_params.csv (v1 kept as pB1_params_v1.csv)

In [4]:
v1 = pd.read_csv(config.PB1_PARAMS_V1)
new = v1.assign(ci_method="group bootstrap")
for r in style.itertuples():
    i = (new["interface"] == r.interface) & (new["parameter"] == r.parameter)
    new.loc[i, ["estimate", "ci_low", "ci_high", "source"]] = [r.estimate, r.ci_low, r.ci_high,
                                                               "per-attempt mixture (build fix 1)"]
i = (new["interface"] == "old_cl") & (new["parameter"] == "rho")
new.loc[i, ["ci_low", "ci_high", "ci_method"]] = [rho_ci[0], rho_ci[1],
                                                  "parametric (1,000 refits, seed 573); known limitation: 14 players, 2 groups"]
new.to_csv(config.PB1_PARAMS, index=False)
cmp = v1.set_index(["interface", "parameter"])[["estimate", "ci_low", "ci_high"]].add_prefix("v1 ").join(
    new.set_index(["interface", "parameter"])[["estimate", "ci_low", "ci_high", "ci_method"]])
cmp.round(4)

v1 estimate  v1 ci_low  v1 ci_high  estimate  ci_low  ci_high                                          ci_method
interface parameter                                                                                                                   
lab_v09   pi               0.4286     0.3814      0.4780    0.3404  0.3031   0.3787                                    group bootstrap
          rho              0.1819     0.1034      0.2689    0.1701  0.1069   0.2420                                    group bootstrap
          q_sg             0.4663     0.3997      0.5256    0.4768  0.4173   0.5311                                    group bootstrap
          q_E              0.4462     0.4063      0.4834    0.4462  0.4063   0.4834                                    group bootstrap
          lam_resume       1.7681     1.4773      2.0651    1.7681  1.4773   2.0651                                    group bootstrap
old_cl    pi               0.2841     0.1667      0.4304    0.1618  0.0769   0.3044                                    group bootstrap
          rho              0.4000     0.2127      0.4000    0.1898  0.0000   1.0000  parametric (1,000 refits, seed 573); known lim...
          q_sg             0.1389     0.1121      0.2233    0.1406  0.1207   0.2059                                    group bootstrap
          q_E              0.2538     0.2523      0.2547    0.2538  0.2523   0.2547                                    group bootstrap
          lam_resume       1.7356     1.5600      2.1657    1.7356  1.5600   2.1657                                    group bootstrap
cc        pi               0.3783     0.3377      0.4216    0.3202  0.2819   0.3576                                    group bootstrap
          rho              0.4161     0.3627      0.4701    0.3935  0.3476   0.4420                                    group bootstrap
          q_sg             0.4914     0.4094      0.5583    0.5014  0.4241   0.5674                                    group bootstrap
          q_E              0.5033     0.4441      0.5661    0.5033  0.4441   0.5661                                    group bootstrap
          lam_resume       2.5491     2.1826      2.9911    2.5491  2.1826   2.9911                                    group bootstrap

**How to read it.** pi is now the mean smooth propensity per crossing attempt, rho the spread of that propensity between players, and q_sg the stop chance per step in a stop-and-go attempt, all fit on every attempt, not only completed crossings. q_E and lam_resume are unchanged from v1. Old CL rho keeps its estimate, with a parametric CI, since 2 groups cannot give a usable group bootstrap.

## 3. Build check again: B1 (new parameters) on G6train, R = 2, seed 573

In [5]:
import time

print({name: sim.b1_params(name) for name in motion.INTERFACES})
G6 = design.design("G6train")
start = time.time()
rounds = design.simulate_rounds(G6, "B1", R=2, seed=573)              # pB1_params.csv (build fix 1)
sim_runs = [run for r in range(2) for run in design.copy_runs(G6, rounds, r, 573)]
est, used = model_b1.estimates(sim_runs)                              # pi, rho, q_sg: per-attempt code
seats_sim, _ = motion.motion_table(sim_runs)
lam = seats_sim.groupby("interface")[["resumes", "halted_s"]].sum()
est["lam_resume"] = lam["resumes"] / lam["halted_s"]
print(len(sim_runs), "rounds |", len(model_b1.attempt_table(used["steps"])), "simulated attempts |",
      f"{(time.time() - start) / 60:.1f} min")
est.round(4)

{'lab_v09': {'pi': 0.340421843511773, 'rho': 0.1701056690118052, 'q_sg': 0.4768497705617164, 'q_E': 0.4461799021452766}, 'old_cl': {'pi': 0.1617851482600639, 'rho': 0.1898244755201663, 'q_sg': 0.1406044537876766, 'q_E': 0.2538192928852029}, 'cc': {'pi': 0.3202227263592135, 'rho': 0.3935301479188935, 'q_sg': 0.5013678366415926, 'q_E': 0.5033425088478175}}


488 rounds | 24791 simulated attempts | 5.7 min


,pi,rho,q_sg,q_E,resume_C,resume_E,resume_all,lam_resume
interface,,,,,,,,
lab_v09,0.3404,0.1786,0.4762,0.4433,0.1489,0.1535,0.1505,1.7656
old_cl,0.6259,0.0000,0.1450,0.2541,0.1797,0.1772,0.1779,1.7786
cc,0.3528,0.4021,0.5006,0.5072,0.0827,0.0840,0.0831,2.5543


In [6]:
rec = new.merge(est.reset_index().melt(id_vars="interface", var_name="parameter", value_name="re_estimate"),
                on=["interface", "parameter"], how="left")
rec["pass"] = (rec["re_estimate"] >= rec["ci_low"]) & (rec["re_estimate"] <= rec["ci_high"])
rec = rec[["interface", "parameter", "estimate", "ci_low", "ci_high", "ci_method", "re_estimate", "pass"]].rename(
    columns={"estimate": "true (human fit)"})
rec.to_csv(config.PB1_RECOVERY, index=False)
print(f"PASS {int(rec['pass'].sum())} of {len(rec)}  (v1: {int(pd.read_csv(config.PB1_RECOVERY_V1)['pass'].sum())} of 15)")
rec.round(4)

PASS 14 of 15  (v1: 12 of 15)


,interface,parameter,true (human fit),ci_low,ci_high,ci_method,re_estimate,pass
0,lab_v09,pi,0.3404,0.3031,0.3787,group bootstrap,0.3404,True
1,lab_v09,rho,0.1701,0.1069,0.2420,group bootstrap,0.1786,True
2,lab_v09,q_sg,0.4768,0.4173,0.5311,group bootstrap,0.4762,True
3,lab_v09,q_E,0.4462,0.4063,0.4834,group bootstrap,0.4433,True
4,lab_v09,lam_resume,1.7681,1.4773,2.0651,group bootstrap,1.7656,True
5,old_cl,pi,0.1618,0.0769,0.3044,group bootstrap,0.6259,False
6,old_cl,rho,0.1898,0.0000,1.0000,"parametric (1,000 refits, seed 573); known lim...",0.0000,True
7,old_cl,q_sg,0.1406,0.1207,0.2059,group bootstrap,0.1450,True
8,old_cl,q_E,0.2538,0.2523,0.2547,group bootstrap,0.2541,True
9,old_cl,lam_resume,1.7356,1.5600,2.1657,group bootstrap,1.7786,True


**How to read it.** B1 was simulated with the new parameters on every G-train round (2 copies). pi, rho and q_sg were then re-estimated from the simulated logs with the per-attempt code, and q_E and lam_resume as in v1. PASS = the re-estimate falls inside the human 95% CI (old CL rho: the parametric CI).

## Build check verdict (written after the results; nothing refit, no number changed)

**PASS 14 of 15** (v1: 12 of 15). pi and rho on lab v09 and CC are now recovered (lab v09 pi 0.3404 vs. 0.3404; CC 0.3528 in [0.282, 0.358]), and every stop and resume chance still passes. Old CL rho passes, but only because its parametric CI is [0, 1].

**The miss: old CL pi (0.626 vs. CI [0.077, 0.304]) is a numerical failure of the estimator, not of B1.** Checked on the same simulated runs (same seed, deterministic), with scratch code only:
- From the default start, Nelder-Mead walks rho down to 1.6e-38, where a, b ~ 1e37. There `betaln(a + s, b + t) - betaln(a, b)` cancels catastrophically: the code reports log-likelihood -19,093, while the true value there is -20,381, far below the interior optimum (-19,711).
- From other starts the same code gives pi 0.138, rho 0.134, q_sg 0.143 (all inside the CIs). The same likelihood written with log rising factorials (s, t are integers: log B(a + s, b + t) - log B(a, b) = sum log(a + i) + sum log(b + i) - sum log(a + b + i)) has no cancellation. From the default start it gives pi 0.136, q_sg 0.143 (rho at the boundary, 0).
- The human fits are identical with both forms (to 4 decimals), so pB1_params.csv's estimates are unaffected. The old CL parametric CI is: 62 of the first 100 refits collapse to rho ~ 0 with the current form, 25 with the stable one. Either way it spans about [0, 1], so old CL rho is not identified from 14 players.

**Proposal (not applied):** replace the betaln difference in `model_b1.mixture_loglik` by the log rising factorials (or bound rho >= 1e-4). Then recompute the old CL parametric CI and rerun the recovery, which is a third recovery run and needs your go-ahead. Old CL rho stays a known limitation (not identified).